<a href="https://colab.research.google.com/github/varsha-pv/Airbnb-AI-Search-Assistant/blob/main/Airbnb_AI_Search_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏠 Airbnb Chatbot — Groq + OpenAI GPT-OSS-120B

A refined Google Colab chatbot that uses **OpenAI GPT-OSS-120B through Groq**, public web search for indexed Airbnb pages, saved searches, listing details, comparisons, and price alerts.


**You'll need a Groq API key.**


## 1. Install dependencies

In [1]:
# Install compatible packages for the Colab app.
!pip install -q -U "gradio>=4.44,<7" groq ddgs
print("Dependencies installed. Restart the runtime if Gradio was already imported before this cell.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 36.9 MB/s eta 0:00:00
Dependencies installed. Restart the runtime if Gradio was already imported before this cell.


## 2. Set your Groq API key

**For "Run All" to work with zero manual typing:** click the key icon (🔑) in the left sidebar of Colab → **Secrets** → add a secret named `GROQ_API_KEY` with your key from https://console.groq.com/keys → toggle **Notebook access** on. This cell reads it automatically.

If you skip that, it falls back to a manual prompt (which pauses "Run All" until you type your key into the box that appears).

In [2]:
import os
from getpass import getpass

def _get_secret(name: str) -> str:

    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    return getpass(f"Enter your {name} (or set it in Colab Secrets to skip this prompt): ")

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = _get_secret("GROQ_API_KEY")

GROQ_MODEL = "openai/gpt-oss-120b"
print("Using model:", GROQ_MODEL)


Using model: openai/gpt-oss-120b


## 3. Data models
Plain dataclasses for listings, saved searches, and price alerts.

In [3]:
from __future__ import annotations
from dataclasses import dataclass, field, asdict
from datetime import datetime
from typing import Optional
import uuid


@dataclass
class Listing:
    listing_id: str
    name: str
    url: str
    price_per_night: Optional[float] = None
    currency: str = "USD"
    rating: Optional[float] = None
    review_count: Optional[int] = None
    room_type: Optional[str] = None
    bedrooms: Optional[int] = None
    beds: Optional[int] = None
    baths: Optional[float] = None
    latitude: Optional[float] = None
    longitude: Optional[float] = None
    is_superhost: bool = False
    snippet: Optional[str] = None

    def to_dict(self) -> dict:
        return asdict(self)


@dataclass
class ListingDetails(Listing):
    description: Optional[str] = None
    amenities: list = field(default_factory=list)
    house_rules: list = field(default_factory=list)
    highlights: list = field(default_factory=list)
    host_name: Optional[str] = None
    max_guests: Optional[int] = None


@dataclass
class SavedSearch:
    search_id: str
    name: str
    location: str
    checkin: Optional[str] = None
    checkout: Optional[str] = None
    adults: int = 1
    children: int = 0
    infants: int = 0
    pets: int = 0
    min_price: Optional[float] = None
    max_price: Optional[float] = None
    room_type: Optional[str] = None
    bedrooms: Optional[int] = None
    created_at: str = field(default_factory=lambda: datetime.utcnow().isoformat())

    @staticmethod
    def new(**kwargs) -> "SavedSearch":
        return SavedSearch(search_id=str(uuid.uuid4())[:8], **kwargs)

    def to_dict(self) -> dict:
        return asdict(self)


@dataclass
class PriceAlert:
    alert_id: str
    listing_id: str
    target_price: float
    checkin: Optional[str] = None
    checkout: Optional[str] = None
    adults: int = 1
    last_checked_price: Optional[float] = None
    last_checked_at: Optional[str] = None
    triggered: bool = False
    created_at: str = field(default_factory=lambda: datetime.utcnow().isoformat())

    @staticmethod
    def new(**kwargs) -> "PriceAlert":
        return PriceAlert(alert_id=str(uuid.uuid4())[:8], **kwargs)

    def to_dict(self) -> dict:
        return asdict(self)

print("Models loaded.")


Models loaded.


## 4. Public Web Search

This version uses public web search to discover indexed Airbnb pages instead of repeatedly scraping Airbnb directly.


In [4]:
from ddgs import DDGS
import hashlib
import re
from datetime import datetime


def _valid_date_range(checkin=None, checkout=None):
    if not checkin and not checkout:
        return
    try:
        ci = datetime.strptime(checkin, "%Y-%m-%d").date() if checkin else None
        co = datetime.strptime(checkout, "%Y-%m-%d").date() if checkout else None
    except ValueError as exc:
        raise ValueError("Dates must use YYYY-MM-DD format.") from exc
    if ci and co and co <= ci:
        raise ValueError("Checkout must be after check-in.")


def _listing_id_from_url(url):
    m = re.search(r"/rooms/(?:[^/?#]+/)?(\d+)", url or "")
    if m:
        return m.group(1)
    return "web-" + hashlib.sha1((url or "").encode()).hexdigest()[:10]


def _extract_number(pattern, text, cast=float):
    m = re.search(pattern, text or "", re.I)
    if not m:
        return None
    try:
        return cast(m.group(1))
    except Exception:
        return None


def _extract_price(text):
    patterns = [
        r"₹\s*([\d,]+(?:\.\d+)?)",
        r"(?:INR|Rs\.?)[\s]*([\d,]+(?:\.\d+)?)",
        r"([\d,]+(?:\.\d+)?)\s*(?:INR|per night|/night)",
    ]
    for p in patterns:
        value = _extract_number(p, text, lambda x: float(x.replace(",", "")))
        if value is not None:
            return value
    return None


def _extract_bedrooms(text):
    return _extract_number(r"(\d+)\s*(?:bedroom|bedrooms|bhk|bed\s*room)", text, int)


def _result_to_listing(row):
    url = row.get("href") or row.get("url") or ""
    title = row.get("title") or "Untitled Airbnb result"
    snippet = row.get("body") or row.get("snippet") or ""
    combined = f"{title} {snippet}"
    rating = _extract_number(r"(\d(?:\.\d)?)\s*(?:out of 5|/5)", combined)
    reviews = _extract_number(
        r"([\d,]+)\s*(?:reviews|review)",
        combined,
        lambda x: int(x.replace(",", "")),
    )
    return Listing(
        listing_id=_listing_id_from_url(url),
        name=title,
        url=url,
        price_per_night=_extract_price(combined),
        rating=rating,
        review_count=reviews,
        bedrooms=_extract_bedrooms(combined),
        snippet=snippet,
    )


def web_search_airbnb(
    location, checkin=None, checkout=None, adults=1, children=0, infants=0,
    pets=0, min_price=None, max_price=None, room_type=None, bedrooms=None,
    max_results=8
):
    _valid_date_range(checkin, checkout)
    if not location or not str(location).strip():
        raise ValueError("A location is required.")

    max_results = max(1, min(int(max_results or 8), 10))
    room_text = room_type.replace("_", " ") if room_type else ""

    query1 = (
        f"site:airbnb.com/rooms Airbnb {location} "
        f"{bedrooms or ''} bedroom {room_text} {adults or 1} guests "
        f"{max_price or ''} {checkin or ''} {checkout or ''}"
    )
    query2 = (
        f"site:airbnb.com/rooms Airbnb {location} "
        f"{bedrooms or ''} bedroom {room_text}"
    )

    rows, seen = [], set()
    with DDGS() as ddgs:
        for query in (query1, query2):
            try:
                batch = list(ddgs.text(query, max_results=max_results * 4))
            except Exception as exc:
                if not rows:
                    raise RuntimeError(f"Public web search failed: {exc}") from exc
                batch = []

            for row in batch:
                url = row.get("href") or row.get("url") or ""
                if "airbnb." not in url.lower() or url in seen:
                    continue
                seen.add(url)
                rows.append(row)
                if len(rows) >= max_results * 2:
                    break
            if len(rows) >= max_results:
                break

    listings = []
    for row in rows:
        listing = _result_to_listing(row)
        if "/rooms/" not in listing.url.lower() and listings:
            continue

        if bedrooms and listing.bedrooms is not None and listing.bedrooms < int(bedrooms):
            continue
        if max_price is not None and listing.price_per_night is not None:
            if listing.price_per_night > float(max_price):
                continue

        listings.append(listing)
        if len(listings) >= max_results:
            break

    return {
        "query": query1,
        "results": [x.to_dict() for x in listings],
        "count": len(listings),
        "search_method": "public web index",
        "filters": {
            "location": location, "checkin": checkin, "checkout": checkout,
            "adults": adults, "bedrooms": bedrooms, "room_type": room_type,
            "min_price": min_price, "max_price": max_price,
        },
        "warning": (
            "These are public web-indexed results, not guaranteed live availability "
            "or final checkout pricing. Verify on the Airbnb page."
        ),
    }


def web_search_listing(listing_id, max_results=5):
    query = f'site:airbnb.com/rooms "{listing_id}"'
    with DDGS() as ddgs:
        rows = list(ddgs.text(query, max_results=max_results))

    return {
        "listing_id": str(listing_id),
        "results": [
            _result_to_listing(row).to_dict()
            for row in rows
            if "airbnb." in (row.get("href") or row.get("url") or "").lower()
        ],
    }


print("Public web-search provider loaded.")


Public web-search provider loaded.


## 5. Storage
Saved searches and price alerts persist to JSON files under `/content/data`. Remember Colab's disk resets when the runtime recycles — mount Google Drive if you want this to survive across sessions.

In [5]:
import json as _json, threading
from pathlib import Path

DATA_DIR = Path("/content/data")
SEARCHES_FILE = DATA_DIR / "saved_searches.json"
ALERTS_FILE = DATA_DIR / "price_alerts.json"
_lock = threading.Lock()

def _ensure_data_dir():
    DATA_DIR.mkdir(parents=True, exist_ok=True)

def _load(path: Path):
    _ensure_data_dir()
    if not path.exists():
        return []
    try:
        return _json.loads(path.read_text())
    except (_json.JSONDecodeError, OSError):
        return []

def _save(path: Path, items):
    _ensure_data_dir()
    path.write_text(_json.dumps(items, indent=2))


class Storage:
    def add_search(self, search: SavedSearch) -> SavedSearch:
        with _lock:
            items = _load(SEARCHES_FILE)
            items.append(search.to_dict())
            _save(SEARCHES_FILE, items)
        return search

    def list_searches(self):
        with _lock:
            return _load(SEARCHES_FILE)

    def get_search(self, search_id: str):
        for item in self.list_searches():
            if item["search_id"] == search_id:
                return item
        return None

    def delete_search(self, search_id: str) -> bool:
        with _lock:
            items = _load(SEARCHES_FILE)
            new_items = [i for i in items if i["search_id"] != search_id]
            deleted = len(new_items) != len(items)
            _save(SEARCHES_FILE, new_items)
        return deleted

    def add_alert(self, alert: PriceAlert) -> PriceAlert:
        with _lock:
            items = _load(ALERTS_FILE)
            items.append(alert.to_dict())
            _save(ALERTS_FILE, items)
        return alert

    def list_alerts(self):
        with _lock:
            return _load(ALERTS_FILE)

    def get_alert(self, alert_id: str):
        for item in self.list_alerts():
            if item["alert_id"] == alert_id:
                return item
        return None

    def update_alert(self, alert_id: str, **fields):
        with _lock:
            items = _load(ALERTS_FILE)
            updated = None
            for item in items:
                if item["alert_id"] == alert_id:
                    item.update(fields)
                    updated = item
                    break
            _save(ALERTS_FILE, items)
        return updated

    def delete_alert(self, alert_id: str) -> bool:
        with _lock:
            items = _load(ALERTS_FILE)
            new_items = [i for i in items if i["alert_id"] != alert_id]
            deleted = len(new_items) != len(items)
            _save(ALERTS_FILE, new_items)
        return deleted

storage = Storage()
print("Storage ready at", DATA_DIR)


Storage ready at /content/data


## 6. Business logic\n\nAll online operations use the fixed public-web provider. Saved searches and alerts use local JSON storage.\n

In [6]:
from datetime import datetime, timezone


def do_search(
    location, checkin=None, checkout=None, adults=1, children=0, infants=0,
    pets=0, min_price=None, max_price=None, room_type=None, bedrooms=None,
    max_results=8
):
    return web_search_airbnb(
        location=location, checkin=checkin, checkout=checkout,
        adults=adults, children=children, infants=infants, pets=pets,
        min_price=min_price, max_price=max_price, room_type=room_type,
        bedrooms=bedrooms, max_results=max_results
    )


def do_listing_details(listing_id, checkin=None, checkout=None, adults=1):
    _valid_date_range(checkin, checkout)
    data = web_search_listing(listing_id)
    if not data["results"]:
        return {
            "listing_id": str(listing_id),
            "found": False,
            "message": "No public indexed result was found for this listing ID."
        }

    top = data["results"][0]
    return {
        "listing_id": str(listing_id),
        "found": True,
        "name": top.get("name"),
        "url": top.get("url"),
        "price_per_night": top.get("price_per_night"),
        "rating": top.get("rating"),
        "review_count": top.get("review_count"),
        "bedrooms": top.get("bedrooms"),
        "snippet": top.get("snippet"),
        "warning": "Verify live availability, taxes, fees and final checkout price on Airbnb."
    }


def do_compare_listings(listing_ids, checkin=None, checkout=None, adults=1):
    if not listing_ids or len(listing_ids) < 2:
        raise ValueError("Provide at least two listing IDs.")

    rows = []
    for listing_id in listing_ids[:6]:
        try:
            rows.append(do_listing_details(listing_id, checkin, checkout, adults))
        except Exception as exc:
            rows.append({"listing_id": str(listing_id), "error": str(exc)})

    priced = [r for r in rows if isinstance(r.get("price_per_night"), (int, float))]
    rated = [r for r in rows if isinstance(r.get("rating"), (int, float))]

    return {
        "listings": rows,
        "cheapest_listing_id": min(priced, key=lambda x: x["price_per_night"])["listing_id"] if priced else None,
        "best_rated_listing_id": max(rated, key=lambda x: x["rating"])["listing_id"] if rated else None,
    }


def do_save_search(
    name, location, checkin=None, checkout=None, adults=1, children=0,
    infants=0, pets=0, min_price=None, max_price=None, room_type=None, bedrooms=None
):
    _valid_date_range(checkin, checkout)
    search = SavedSearch.new(
        name=name, location=location, checkin=checkin, checkout=checkout,
        adults=adults, children=children, infants=infants, pets=pets,
        min_price=min_price, max_price=max_price, room_type=room_type,
        bedrooms=bedrooms
    )
    storage.add_search(search)
    return search.to_dict()


def do_list_saved_searches():
    return {"searches": storage.list_searches()}


def do_run_saved_search(search_id, max_results=8):
    saved = storage.get_search(search_id)
    if not saved:
        raise ValueError(f"No saved search with id {search_id}")
    return do_search(
        location=saved["location"], checkin=saved.get("checkin"),
        checkout=saved.get("checkout"), adults=saved.get("adults", 1),
        children=saved.get("children", 0), infants=saved.get("infants", 0),
        pets=saved.get("pets", 0), min_price=saved.get("min_price"),
        max_price=saved.get("max_price"), room_type=saved.get("room_type"),
        bedrooms=saved.get("bedrooms"), max_results=max_results
    )


def do_delete_saved_search(search_id):
    return {"deleted": storage.delete_search(search_id)}


def do_create_price_alert(listing_id, target_price, checkin=None, checkout=None, adults=1):
    _valid_date_range(checkin, checkout)
    alert = PriceAlert.new(
        listing_id=str(listing_id), target_price=float(target_price),
        checkin=checkin, checkout=checkout, adults=adults
    )
    storage.add_alert(alert)
    return alert.to_dict()


def do_list_price_alerts():
    return {"alerts": storage.list_alerts()}


def do_delete_price_alert(alert_id):
    return {"deleted": storage.delete_alert(alert_id)}


def do_check_price_alerts():
    results = []
    for alert in storage.list_alerts():
        try:
            details = do_listing_details(
                alert["listing_id"], alert.get("checkin"),
                alert.get("checkout"), alert.get("adults", 1)
            )
            current = details.get("price_per_night")
            triggered = (
                isinstance(current, (int, float))
                and current <= float(alert["target_price"])
            )
            storage.update_alert(
                alert["alert_id"],
                last_checked_price=current,
                last_checked_at=datetime.now(timezone.utc).isoformat(),
                triggered=triggered
            )
            results.append({
                "alert_id": alert["alert_id"],
                "listing_id": alert["listing_id"],
                "target_price": alert["target_price"],
                "current_price": current,
                "triggered": triggered
            })
        except Exception as exc:
            results.append({
                "alert_id": alert["alert_id"],
                "listing_id": alert["listing_id"],
                "error": str(exc)
            })
    return results


print("Business logic ready.")


Business logic ready.


## 7. Web search + listing extraction

Search snippets are converted into structured listing information. Missing values are left unknown rather than guessed.


In [7]:
# The web-search implementation is defined in the provider cell above.
# This cell is intentionally empty so an older do_search() cannot override it.
print("Web-search implementation is locked to the fixed provider.")


Web-search implementation is locked to the fixed provider.


## 8. Tool schema for Groq function-calling
Each entry pairs a JSON-schema spec (what GPT-OSS-120B sees) with the real Python callable (what actually runs).

In [8]:
TOOLS = [
    {"type": "function", "function": {
        "name": "airbnb_search",
        "description": "Search public web indexes for Airbnb listing pages. Use for accommodation searches.",
        "parameters": {"type": "object", "properties": {
            "location": {"type": "string"},
            "checkin": {"type": "string"},
            "checkout": {"type": "string"},
            "adults": {"type": "integer", "default": 1},
            "children": {"type": "integer", "default": 0},
            "infants": {"type": "integer", "default": 0},
            "pets": {"type": "integer", "default": 0},
            "min_price": {"type": "number"},
            "max_price": {"type": "number"},
            "room_type": {"type": "string", "enum": ["entire_home", "private_room", "shared_room", "hotel_room"]},
            "bedrooms": {"type": "integer", "minimum": 1},
            "max_results": {"type": "integer", "minimum": 1, "maximum": 10, "default": 8}
        }, "required": ["location"], "additionalProperties": False}
    }},
    {"type": "function", "function": {
        "name": "airbnb_listing_details",
        "description": "Look up public indexed information for one Airbnb listing ID.",
        "parameters": {"type": "object", "properties": {
            "listing_id": {"type": "string"},
            "checkin": {"type": "string"},
            "checkout": {"type": "string"},
            "adults": {"type": "integer", "default": 1}
        }, "required": ["listing_id"], "additionalProperties": False}
    }},
    {"type": "function", "function": {
        "name": "compare_listings",
        "description": "Compare up to six indexed Airbnb listing IDs.",
        "parameters": {"type": "object", "properties": {
            "listing_ids": {"type": "array", "items": {"type": "string"}, "minItems": 2, "maxItems": 6},
            "checkin": {"type": "string"},
            "checkout": {"type": "string"},
            "adults": {"type": "integer", "default": 1}
        }, "required": ["listing_ids"], "additionalProperties": False}
    }},
    {"type": "function", "function": {
        "name": "save_search",
        "description": "Save a search configuration. Preserve all filters including bedrooms.",
        "parameters": {"type": "object", "properties": {
            "name": {"type": "string"}, "location": {"type": "string"},
            "checkin": {"type": "string"}, "checkout": {"type": "string"},
            "adults": {"type": "integer", "default": 1}, "children": {"type": "integer", "default": 0},
            "infants": {"type": "integer", "default": 0}, "pets": {"type": "integer", "default": 0},
            "min_price": {"type": "number"}, "max_price": {"type": "number"},
            "room_type": {"type": "string", "enum": ["entire_home", "private_room", "shared_room", "hotel_room"]},
            "bedrooms": {"type": "integer", "minimum": 1}
        }, "required": ["name", "location"], "additionalProperties": False}
    }},
    {"type": "function", "function": {
        "name": "list_saved_searches",
        "description": "List all saved searches.",
        "parameters": {"type": "object", "properties": {}, "additionalProperties": False}
    }},
    {"type": "function", "function": {
        "name": "run_saved_search",
        "description": "Run a saved search by ID.",
        "parameters": {"type": "object", "properties": {
            "search_id": {"type": "string"},
            "max_results": {"type": "integer", "minimum": 1, "maximum": 10, "default": 8}
        }, "required": ["search_id"], "additionalProperties": False}
    }},
    {"type": "function", "function": {
        "name": "delete_saved_search",
        "description": "Delete a saved search by ID.",
        "parameters": {"type": "object", "properties": {
            "search_id": {"type": "string"}
        }, "required": ["search_id"], "additionalProperties": False}
    }},
    {"type": "function", "function": {
        "name": "create_price_alert",
        "description": "Create a price alert for a listing ID.",
        "parameters": {"type": "object", "properties": {
            "listing_id": {"type": "string"}, "target_price": {"type": "number"},
            "checkin": {"type": "string"}, "checkout": {"type": "string"},
            "adults": {"type": "integer", "default": 1}
        }, "required": ["listing_id", "target_price"], "additionalProperties": False}
    }},
    {"type": "function", "function": {
        "name": "list_price_alerts",
        "description": "List all price alerts.",
        "parameters": {"type": "object", "properties": {}, "additionalProperties": False}
    }},
    {"type": "function", "function": {
        "name": "check_price_alerts",
        "description": "Re-check indexed web prices for all saved alerts.",
        "parameters": {"type": "object", "properties": {}, "additionalProperties": False}
    }},
    {"type": "function", "function": {
        "name": "delete_price_alert",
        "description": "Delete a price alert by ID.",
        "parameters": {"type": "object", "properties": {
            "alert_id": {"type": "string"}
        }, "required": ["alert_id"], "additionalProperties": False}
    }},
]

TOOL_IMPL = {
    "airbnb_search": do_search,
    "airbnb_listing_details": do_listing_details,
    "compare_listings": do_compare_listings,
    "save_search": do_save_search,
    "list_saved_searches": do_list_saved_searches,
    "run_saved_search": do_run_saved_search,
    "delete_saved_search": do_delete_saved_search,
    "create_price_alert": do_create_price_alert,
    "list_price_alerts": do_list_price_alerts,
    "check_price_alerts": do_check_price_alerts,
    "delete_price_alert": do_delete_price_alert,
}

assert set(x["function"]["name"] for x in TOOLS) == set(TOOL_IMPL)
print(f"{len(TOOLS)} tools registered and matched.")


11 tools registered and matched.


## 9. GPT-OSS-120B chat loop

The agent performs at most one local tool call and then disables tools for the final answer.


In [9]:
import json
import logging
from groq import Groq

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("airbnb-chatbot")

groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])
GROQ_MODEL = "openai/gpt-oss-120b"

SYSTEM_PROMPT = """You are a concise Airbnb search assistant.
Use airbnb_search exactly once for an accommodation search.
After a tool result, answer directly and do not call another tool in that turn.
Do not invent prices, ratings, availability, amenities, or URLs.
Interpret 2 BHK as 2 bedrooms and 30k as 30000 INR.
Reject impossible dates instead of searching.
Show useful result details and URLs when available.
Clearly say that indexed web data is not guaranteed live availability or final checkout pricing."""

MAX_COMPLETION_TOKENS = 2048


def _clean_history(history):
    """Normalize Gradio history and remove unsupported metadata."""
    clean = []

    for item in history or []:

        # Newer Gradio message-dict format
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content")

            if role in ("user", "assistant") and content is not None:
                clean.append({
                    "role": role,
                    "content": str(content)
                })

        # Older Gradio tuple format: (user_message, assistant_message)
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            user_msg, assistant_msg = item

            if user_msg is not None and str(user_msg).strip():
                clean.append({
                    "role": "user",
                    "content": str(user_msg)
                })

            if assistant_msg is not None and str(assistant_msg).strip():
                clean.append({
                    "role": "assistant",
                    "content": str(assistant_msg)
                })

    return clean


def _model_call(messages, use_tools):
    kwargs = {
        "model": GROQ_MODEL,
        "messages": messages,
        "temperature": 0.3,
        "max_completion_tokens": MAX_COMPLETION_TOKENS,
        "reasoning_effort": "low",
        "include_reasoning": False,
    }
    if use_tools:
        kwargs["tools"] = TOOLS
        kwargs["tool_choice"] = "auto"
        # GPT-OSS 120B does not support parallel local tool use.
        kwargs["parallel_tool_calls"] = False
    # IMPORTANT: when tools are disabled, do not send tools=[] or tool_choice.
    # GPT-OSS can otherwise still emit a tool call and Groq rejects the request.
    return groq_client.chat.completions.create(**kwargs)


def chat(message, history=None):
    clean_history = _clean_history(history)
    messages = (
        [{"role": "system", "content": SYSTEM_PROMPT}]
        + clean_history
        + [{"role": "user", "content": str(message)}]
    )

    try:
        first = _model_call(messages, use_tools=True)
        assistant_msg = first.choices[0].message

        if not assistant_msg.tool_calls:
            reply = assistant_msg.content or "I couldn't generate a response."
            return {
                "reply": reply,
                "tool_calls_made": [],
                "history": clean_history + [
                    {"role": "user", "content": str(message)},
                    {"role": "assistant", "content": reply}
                ],
            }

        # Exactly one tool call per turn.
        tc = assistant_msg.tool_calls[0]
        fn_name = tc.function.name

        try:
            args = json.loads(tc.function.arguments or "{}")
        except Exception as exc:
            tool_result = {"error": f"Invalid tool arguments: {exc}"}
        else:
            impl = TOOL_IMPL.get(fn_name)
            if impl is None:
                tool_result = {"error": f"Unknown tool: {fn_name}"}
            else:
                try:
                    tool_result = impl(**args)
                except Exception as exc:
                    logger.exception("Tool %s failed", fn_name)
                    tool_result = {"error": str(exc)}

        messages.append({
            "role": "assistant",
            "content": assistant_msg.content or "",
            "tool_calls": [tc.model_dump()]
        })
        messages.append({
            "role": "tool",
            "tool_call_id": tc.id,
            "name": fn_name,
            "content": json.dumps(tool_result, default=str)
        })

        # Final pass has no tools, so the model cannot loop/search again.
        final = _model_call(messages, use_tools=False)
        reply = final.choices[0].message.content or "I received the tool result but couldn't summarize it."

        return {
            "reply": reply,
            "tool_calls_made": [fn_name],
            "history": clean_history + [
                {"role": "user", "content": str(message)},
                {"role": "assistant", "content": reply}
            ],
        }

    except Exception as exc:
        logger.exception("Chat failed")
        raise RuntimeError(f"Groq request failed: {exc}") from exc


print("GPT-OSS-120B chat loop ready.")


GPT-OSS-120B chat loop ready.


## 10. Try it — including the saved-search fix
Saved searches and price alerts work offline. The saved-search schema now includes `bedrooms`, and the value is preserved when the search is re-run. `airbnb_search` now uses public web search results rather than directly crawling Airbnb, so it can return indexed listing pages without bypassing Airbnb anti-bot controls.

In [10]:
# Deterministic saved-search regression test. No Groq/web request.
test_saved = do_save_search(
    name="__test_goa_bedrooms__",
    location="Goa, India",
    adults=2,
    max_price=6000,
    room_type="entire_home",
    bedrooms=2,
)
assert test_saved["bedrooms"] == 2
saved = storage.get_search(test_saved["search_id"])
assert saved["bedrooms"] == 2
assert do_delete_saved_search(test_saved["search_id"])["deleted"]
print("PASS: saved searches preserve bedrooms.")


PASS: saved searches preserve bedrooms.


/tmp/ipykernel_3423/1339155192.py:55: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  created_at: str = field(default_factory=lambda: datetime.utcnow().isoformat())


In [11]:
# Date validation regression tests.
try:
    _valid_date_range("2026-09-26", "2026-09-31")
    raise AssertionError("September 31 was accepted.")
except ValueError:
    print("PASS: impossible date rejected.")

_valid_date_range("2026-09-26", "2026-09-30")
print("PASS: valid date accepted.")


PASS: impossible date rejected.
PASS: valid date accepted.


In [12]:
# Gradio metadata regression test.
history_with_metadata = [
    {"role": "user", "content": "2 bhk in Vizag", "metadata": {"id": 1}},
    {"role": "assistant", "content": "Sure", "metadata": {"foo": "bar"}},
]
cleaned = _clean_history(history_with_metadata)
assert cleaned == [
    {"role": "user", "content": "2 bhk in Vizag"},
    {"role": "assistant", "content": "Sure"},
]
print("PASS: Gradio metadata is stripped before Groq.")


PASS: Gradio metadata is stripped before Groq.


In [13]:
# Tool-schema regression test.
assert "bedrooms" in TOOLS[0]["function"]["parameters"]["properties"]
assert "bedrooms" in TOOLS[3]["function"]["parameters"]["properties"]
assert set(TOOL_IMPL) == {x["function"]["name"] for x in TOOLS}
print("PASS: tool schemas and implementations match.")


PASS: tool schemas and implementations match.


## 10. Launch the chatbot

All regression tests above are deterministic and do not consume Groq tokens.

The chatbot flow is deliberately simple:
**User → GPT-OSS-120B → ONE tool call → GPT-OSS-120B final answer**

That prevents the repeated-search and timeout problem.


In [14]:
# Optional Groq API smoke test.
# Leave False during Run All. Turn True only when you want to test the API.
RUN_API_SMOKE_TEST = False

if RUN_API_SMOKE_TEST:
    smoke = chat("Say hello in one short sentence.")
    print(smoke["reply"])
else:
    print("API smoke test skipped.")


API smoke test skipped.


In [15]:
# ChatInterface compatibility test
assert isinstance(_clean_history([
    {"role": "user", "content": "hello", "metadata": {"x": 1}},
    {"role": "assistant", "content": "hi", "metadata": {"y": 2}},
]), list)
assert _clean_history([("hello", "hi")]) == [
    {"role": "user", "content": "hello"},
    {"role": "assistant", "content": "hi"},
]
print("PASS: Gradio old/new history formats are supported and metadata is removed.")


PASS: Gradio old/new history formats are supported and metadata is removed.


In [ ]:
import gradio as gr
import traceback


def gradio_chat(message, history):
    try:
        result = chat(message, history=history or [])
        reply = result.get("reply", "")
        tools = result.get("tool_calls_made", [])

        if tools:
            reply += f"\n\n*Tool used: {', '.join(tools)}*"

        return reply

    except Exception as exc:
        traceback.print_exc()
        text = str(exc)

        if "429" in text or "rate_limit" in text.lower():
            return (
                "⚠️ Groq rate limit reached. "
                "Please wait a little and try again."
            )

        return f"❌ Something went wrong: {text}"


# Do not use type="messages" here.
# The wrapper normalizes Gradio history before sending it to Groq.
demo = gr.ChatInterface(
    fn=gradio_chat,
    title="Airbnb Agent",
    description=(
        "Search public web-indexed Airbnb pages, save searches, compare results, "
        "and create price alerts. Verify live availability and final checkout price on Airbnb."
    ),
    examples=[
        "Find a 2 bedroom home in Vizag, Andhra Pradesh from 26/09/2026 to 30/09/2026 for 2 adults",
        "Find a 2 BHK in Goa under 6000 INR per night",
        "Save a 2-bedroom entire home in Goa for 2 adults under 6000 INR as Goa weekend",
        "List my saved searches",
    ],
)

print("Starting Gradio...")
demo.launch(share=True, debug=True)


Starting Gradio...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://77fe512687da1fba69.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Notes

- Model: `openai/gpt-oss-120b` via Groq.
- Search source: public web-indexed Airbnb pages.
- The chatbot intentionally performs at most one local tool call per user turn.
- The final GPT-OSS response is requested without any tool configuration.
- Verify live availability, taxes, fees and final checkout price on Airbnb.
- Saved searches and price alerts are stored locally under `/content/data`.
